In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver 
import re

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer clearly and directly."),
    ("human", "{input}")
])

llm = ChatOpenAI()

- `MemorySaver` is for storing memory (state) in same session.
- `InMemorySaver` stores memory (all states) post session.
- `InMemorySaver` stores in RAM. It's not used in production setup. 
- For production, there are `redis`, `postgres` checkpointers.

In [2]:
class JokeState(TypedDict):
  topic: str
  joke: str
  explanation: str

In [3]:
def generate_joke(state: JokeState):
  prompt = f"Generate a joke on the topic {state['topic']}"
  response = llm.invoke(prompt).content
  return {'joke': response}

In [4]:
def generate_explanation(state: JokeState):
  prompt = f"write an explanation for the joke - {state['joke']}"
  response = llm.invoke(prompt).content
  return {'explanation': response}

In [5]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [ ]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic': 'pizza'}, config = config1)

{'topic': 'pizza',
 'joke': "Why did the pizza maker go to therapy? Because he couldn't stop topping himself!",
 'explanation': 'This joke is a play on words that has a double meaning. In the context of making pizza, "topping" refers to adding ingredients such as cheese, sauce, and toppings like pepperoni or vegetables. However, "topping oneself" is also a phrase that means to commit suicide. So, the joke is humorously suggesting that the pizza maker goes to therapy because he can\'t stop adding toppings to the pizza (topping himself), not because he is suicidal. This creates a clever and unexpected punchline that plays on the different meanings of the word "topping."'}

In [ ]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't stop topping himself!", 'explanation': 'This joke is a play on words that has a double meaning. In the context of making pizza, "topping" refers to adding ingredients such as cheese, sauce, and toppings like pepperoni or vegetables. However, "topping oneself" is also a phrase that means to commit suicide. So, the joke is humorously suggesting that the pizza maker goes to therapy because he can\'t stop adding toppings to the pizza (topping himself), not because he is suicidal. This creates a clever and unexpected punchline that plays on the different meanings of the word "topping."'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17d089-afe8-68da-8002-826f3c59a77a'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-11T09:12:13.877671+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': 

In [ ]:
list(workflow.get_state_history(config1)) 

[StateSnapshot(values={'topic': 'pizza', 'joke': "Why did the pizza maker go to therapy? Because he couldn't stop topping himself!", 'explanation': 'This joke is a play on words that has a double meaning. In the context of making pizza, "topping" refers to adding ingredients such as cheese, sauce, and toppings like pepperoni or vegetables. However, "topping oneself" is also a phrase that means to commit suicide. So, the joke is humorously suggesting that the pizza maker goes to therapy because he can\'t stop adding toppings to the pizza (topping himself), not because he is suicidal. This creates a clever and unexpected punchline that plays on the different meanings of the word "topping."'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f17d089-afe8-68da-8002-826f3c59a77a'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-11T09:12:13.877671+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns':

In [9]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic': 'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': "Why did the ravioli break up with the spaghetti? Because they just couldn't mac and cheese it work!",
 'explanation': 'This joke plays on the idea of food items having relationships. In this case, the ravioli and spaghetti are personified as a couple. The joke refers to the popular dish "mac and cheese," suggesting that the ravioli and spaghetti couldn\'t make their relationship work because they couldn\'t "mac and cheese it work" - a pun on the phrase "make it work." The humor comes from the clever wordplay and unexpected association between food items and a romantic relationship.'}

In [10]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the ravioli break up with the spaghetti? Because they just couldn't mac and cheese it work!", 'explanation': 'This joke plays on the idea of food items having relationships. In this case, the ravioli and spaghetti are personified as a couple. The joke refers to the popular dish "mac and cheese," suggesting that the ravioli and spaghetti couldn\'t make their relationship work because they couldn\'t "mac and cheese it work" - a pun on the phrase "make it work." The humor comes from the clever wordplay and unexpected association between food items and a romantic relationship.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17d08b-1527-6ef1-8002-a463ee2ac83e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-11T09:12:51.337981+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17d08b-008e-6726-8001-aa6d4ff54049'

In [13]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': "Why did the ravioli break up with the spaghetti? Because they just couldn't mac and cheese it work!", 'explanation': 'This joke plays on the idea of food items having relationships. In this case, the ravioli and spaghetti are personified as a couple. The joke refers to the popular dish "mac and cheese," suggesting that the ravioli and spaghetti couldn\'t make their relationship work because they couldn\'t "mac and cheese it work" - a pun on the phrase "make it work." The humor comes from the clever wordplay and unexpected association between food items and a romantic relationship.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17d08b-1527-6ef1-8002-a463ee2ac83e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-07-11T09:12:51.337981+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f17d08b-008e-6726-8001-aa6d4ff54049

- `get_state` gives the final state.
- `get_state_history` gives all intermediate steps